# Mission 08: Awesome Agent Skills & Dynamic Skill Loading

이 노트북은 Vercel Labs의 **`npx skills`** CLI와 **awesome-agent-skills** 오픈소스 레지스트리를 연계하여, 깃허브 상에 공개된 검증된 스킬셋을 한 줄의 명령어로 추가하고 우리 에이전트가 이를 자율적으로 학습해 기동하도록 연동하는 실습입니다.

In [ ]:
# 1. 환경 변수 및 패키지 탐색 경로 로드
import sys
import os
from dotenv import load_dotenv

while not os.path.exists("app") and os.getcwd() != "/":
    os.chdir("..")
sys.path.append(os.path.abspath("src"))
sys.path.append(os.path.abspath("."))
load_dotenv(override=True)

print(f"📌 현재 작업 디렉토리: {os.getcwd()}")

### [단계 1] 기본 도구(Bash/FileRead)만을 장착한 에이전트 조립

에이전트에게 처음부터 `jupyter-notebook` 생성 도구를 직접 주입하는 것이 아닙니다. 에이전트에게는 오직 파일 읽기(`file_read`)와 리눅스 셸 명령 실행(`bash_command`)이라는 2가지 기본 도구만 쥐어준 상태에서, 프롬프트에 `Progressive Skill Disclosure` 지침을 주어 로컬 스킬을 학습하여 실행하도록 제어합니다.

In [ ]:
from harness.tools.claude_tools import file_read, bash_command
from app.utils import get_llm
from langchain.agents import create_agent

# 1. 초경량 Gemini Flash 모델 로드 및 오직 2가지 기본 도구만 바인딩
llm = get_llm(model_name="gemini-3.5-flash", temperature=0.0)
base_tools = [file_read, bash_command]

# 2. 에이전트가 로컬 skills 디렉토리를 자율적으로 우선 탐색하도록 가이드
system_instruction = (
    "You are a progressive skill-disclosure agent.\n"
    "You only have 'file_read' and 'bash_command' base tools.\n"
    "Your objective is to solve the user request by discovering and executing custom scripts under the local 'skills/' directory.\n\n"
    "Guidelines:\n"
    "- ALWAYS prioritize searching and utilizing custom scripts under the 'skills/' folder over running raw shell commands directly in 'bash_command'.\n"
    "- If the requested skill does not exist locally, run 'npx -y skills add [owner/repo] --skill [name]' (or with '@' shorthand) via 'bash_command' to download it.\n"
    "- If the tool is installed inside '.agents/skills/[name]', copy it to 'skills/[name]' using cp -r to keep it in the project path.\n"
    "- Once downloaded, read the corresponding 'Skill.md' file under the downloaded folder using 'file_read' to learn how to run the script and its arguments.\n"
    "- Finally, execute the script via 'bash_command' using the exact python path and parameters to fulfill the request.\n"
    "Report the result clearly to the user."
)

# 3. ReAct 에이전트 빌드
from langgraph.checkpoint.memory import MemorySaver
checkpointer = MemorySaver()
skill_agent = create_agent(
    model=llm,
    tools=base_tools,
    system_prompt=system_instruction,
    checkpointer=checkpointer
)

print("🚀 Progressive Skill 에이전트 기동 완료! 자율 탐색 및 실행을 준비합니다.")

### [단계 2] 에이전트의 공개 스킬 자율 다운로드 및 실행 격발

사용자는 단지 1) 스킬 다운로드, 2) 스킬 가이드 확인, 3) 노트북 생성을 지시하고, 에이전트가 `bash_command`로 스킬을 직접 추가한 뒤, `Skill.md`를 찾아 읽고 파이썬 명령어로 노트북을 최종 생성해 내는지 디버거를 기동하여 핑퐁 궤적을 실시간으로 관찰합니다.

In [ ]:
from utils.test_log import stream_and_debug_agent
from langchain_core.messages import HumanMessage

# 에이전트 자율 연동 지시
user_request = (
    "1. 'npx -y skills add openai/skills --skill jupyter-notebook' 명령어를 실행하여 주피터 노트북 스킬을 프로젝트 로컬에 다운로드받아줘.\n"
    "2. 만약 스킬이 '.agents/skills/jupyter-notebook'에 생성되었다면 'skills/jupyter-notebook'으로 통째로 복사(cp -r)해줘.\n"
    "3. 다운로드된 'skills/jupyter-notebook/SKILL.md' 파일을 읽어와 사용법을 파악해줘.\n"
    "4. 해당 스킬 가이드의 지침에 따라 1부터 100까지의 합을 구하는 파이썬 코드가 작성된 주피터 노트북 파일 "
    "'artifacts/sum_1_to_100.ipynb'를 생성해줘."
)

config = {"configurable": {"thread_id": "skills_public_harness_session"}}
inputs = {"messages": [HumanMessage(content=user_request)]}

try:
    # 실시간 에이전트 자율 탐색 및 실행 궤적 추적
    stream_and_debug_agent(skill_agent, inputs, config, agent_name="Progressive Skill 에이전트")
    
    # 최종 결과물 생성 여부 검증
    target_file = "artifacts/sum_1_to_100.ipynb"
    if os.path.exists(target_file):
        print(f"\n🎉 [검증 성공] 에이전트가 npx로 스킬을 직접 다운로드하고 실행해 '{target_file}' 파일을 생성했습니다!")
    else:
        print(f"\n❌ [검증 실패] '{target_file}' 파일이 생성되지 않았습니다. 에이전트 궤적을 점검하세요.")
except Exception as e:
    print(f"❌ 에러 발생: {e}")